# Self-training, part 2: retrain on gold plus executor-verified programs (GPU)

**Question.** Does adding the 873 verified programs from Notebook 20 (777 train records) to the gold
programs improve the Notebook 17 generator on dev (625/883 top beam)?

**What changes.** Only the training data. A fresh LoRA adapter is trained from the same base model with
the same LoRA, optimizer, schedule, seed, epochs and epoch selection as Notebook 17
(`configs/generator_qwen_lora_top10.toml`). Each epoch has every gold example (as before) plus **one**
verified program per record that has any, rotated by epoch, on the facts it was verified with; a record
with 4 verified programs therefore weighs the same as one with 1. Gold labels are not changed.

**Protocol.** The verified file is loaded as saved (SHA-256 checked) and re-audited: train ids only,
none in dev, every program re-verified. Dev decoding uses the 4-beam top-beam pipeline; the 16-sample
confidence decode is skipped. Paired exact McNemar against the Notebook 17 predictions. Adoption needs
fixed > broken with p < 0.05 and is decided outside this notebook. Test is not loaded. The adapter is
saved to its own directory. `FINEVIDENCE_SMOKE_TEST=1` runs a tiny CPU check.

In [ ]:
from __future__ import annotations

import gc
import hashlib
import json
import os
import platform
import random
import subprocess
import time
import tomllib
from pathlib import Path

import numpy as np
import pandas as pd
import peft
import torch
import transformers
from peft import LoraConfig, PeftModel, get_peft_model
from scipy.stats import binomtest
from transformers import AutoModelForCausalLM, AutoTokenizer, get_linear_schedule_with_warmup

from finevidence.data import load_finqa_split
from finevidence.data.acquisition import download_finqa_splits
from finevidence.generation import (
    build_retrieved_examples,
    build_training_examples,
    load_rankings,
    score_program_prediction,
    summarize_scores,
    verify_rankings_file,
)
from finevidence.generation.causal import (
    IGNORE_INDEX,
    INSTRUCTION,
    PROGRAM_MARKER,
    build_causal_example,
    extract_program,
)
from finevidence.generation.self_training import (
    audit_verified_programs,
    load_verified_programs,
    self_training_examples,
)

PROJECT_ROOT = (
    Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
)
SMOKE_TEST = os.environ.get("FINEVIDENCE_SMOKE_TEST") == "1"
REUSE_ADAPTER = False
cfg = tomllib.loads((PROJECT_ROOT / "configs" / "generator_qwen_lora_top10.toml").read_text())
st = cfg["self_training_retrain"]
assert "test" not in (cfg["train_split"], cfg["evaluation_split"])
pretrained = (
    "trl-internal-testing/tiny-Qwen2ForCausalLM-2.5"
    if SMOKE_TEST
    else cfg["model"]["pretrained_name"]
)
epochs = 1 if SMOKE_TEST else cfg["training"]["epochs"]
num_beams = 2 if SMOKE_TEST else cfg["generation"]["num_beams"]
suffix = "-smoke" if SMOKE_TEST else ""
model_dir = Path(os.environ.get("FINEVIDENCE_MODEL_DIR") or PROJECT_ROOT / st["model_dir"])
model_dir = Path(str(model_dir) + suffix)
predictions_dir = PROJECT_ROOT / cfg["output"]["predictions_dir"]
rankings_path = PROJECT_ROOT / cfg["inputs"]["rankings_path"]
# Earlier outputs attached to the kernel (KERNEL_SOURCES) or present locally.
KAGGLE_INPUT = Path("/kaggle/input")


def find_artifact(name: str, sha256: str) -> Path:
    places = [predictions_dir / name]
    if KAGGLE_INPUT.exists():
        places += sorted(KAGGLE_INPUT.rglob(name))
    for path in places:
        if path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() == sha256:
            return path
    raise FileNotFoundError(f"{name} with SHA-256 {sha256[:12]}... not found")


seed = cfg["seed"]
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if not SMOKE_TEST and device.type != "cuda":
    raise RuntimeError(
        "Full training needs a CUDA GPU; set FINEVIDENCE_SMOKE_TEST=1 for a CPU check."
    )
use_fp16 = device.type == "cuda"
commit = subprocess.run(
    ["git", "rev-parse", "HEAD"], capture_output=True, text=True, cwd=PROJECT_ROOT
).stdout.strip()
environment = {
    "commit": commit,
    "python": platform.python_version(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "peft": peft.__version__,
    "device": torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu",
    "smoke_test": SMOKE_TEST,
    "reuse_adapter": REUSE_ADAPTER,
    "pretrained": pretrained,
    "epochs": epochs,
    "num_beams": num_beams,
    "experiment_name": st["experiment_name"],
    "fp16": use_fp16,
    "seed": seed,
}
print(json.dumps(environment, indent=2))

## 1. Verified data, rankings, inputs and self-training audit

In [ ]:
raw_dir = PROJECT_ROOT / cfg["dataset_root"]
print(download_finqa_splits(raw_dir, (cfg["train_split"], cfg["evaluation_split"])))
verify_rankings_file(rankings_path, cfg["inputs"]["rankings_sha256"])
rankings = load_rankings(rankings_path)
train = load_finqa_split(raw_dir / f"{cfg['train_split']}.json")
dev = load_finqa_split(raw_dir / f"{cfg['evaluation_split']}.json")
if SMOKE_TEST:
    train, dev = train[:24], dev[:8]
facts = cfg["training"]["facts_per_input"]
# Notebook 20 output, loaded as saved (no new samples). The audit re-checks every program on train
# and rejects any record id that is not a train id or that appears in dev.
verified_path = find_artifact(st["verified_programs"], st["verified_sha256"])
verified = load_verified_programs(verified_path)
if SMOKE_TEST:  # the smoke run keeps only its first train records
    verified = {k: v for k, v in verified.items() if k in {str(r["id"]) for r in train}}
audit = audit_verified_programs(
    train,
    verified,
    cap=cfg["self_training"]["max_kept_per_record"],
    excluded_ids=[str(record["id"]) for record in dev],
)
if not SMOKE_TEST:
    assert audit == {"records_with_new_program": 777, "new_programs": 873}, audit
gold_programs_before = [record["qa"]["program"] for record in train]
retrieved_examples = build_retrieved_examples(dev, rankings, facts_per_input=facts)
print({"train_records": len(train), "dev_records": len(dev), "test_loaded": False, **audit})

## 2. Tokenization

The prompt is the instruction, the source and the marker `program:`; only the program and the
end-of-sequence token are trained (`finevidence.generation.causal`).

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(pretrained)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
max_source_tokens = cfg["model"]["max_source_tokens"]
marker_ids = tokenizer(PROGRAM_MARKER, add_special_tokens=False).input_ids


def prompt_ids(example) -> list[int]:
    source = tokenizer(INSTRUCTION + example.source, add_special_tokens=False).input_ids
    return [*source[:max_source_tokens], *marker_ids]


def training_features(example) -> tuple[list[int], list[int]]:
    source = tokenizer(INSTRUCTION + example.source, add_special_tokens=False).input_ids
    target = tokenizer(" " + example.target, add_special_tokens=False).input_ids
    return build_causal_example(
        source, marker_ids, target, tokenizer.eos_token_id, max_source_tokens=max_source_tokens
    )


first = build_training_examples(train, facts_per_input=facts, seed=seed)
lengths = np.array([len(training_features(example)[0]) for example in first])
truncated = sum(
    len(tokenizer(INSTRUCTION + e.source, add_special_tokens=False).input_ids) > max_source_tokens
    for e in first
)
print(
    f"train sequence tokens: median {int(np.median(lengths))}, p95 {int(np.percentile(lengths, 95))}, "
    f"max {lengths.max()}; sources truncated: {truncated}/{len(first)}"
)

## 3. Train a fresh LoRA adapter on gold plus verified programs

Same procedure as Notebook 17; the best epoch by dev greedy execution accuracy is saved to `best/`.

In [ ]:
def clock(seconds: float) -> str:
    minutes, seconds = divmod(int(seconds), 60)
    hours, minutes = divmod(minutes, 60)
    return f"{hours}h{minutes:02d}m{seconds:02d}s" if hours else f"{minutes}m{seconds:02d}s"


base = AutoModelForCausalLM.from_pretrained(
    pretrained, torch_dtype=torch.float16 if use_fp16 else torch.float32
).to(device)
base.gradient_checkpointing_enable()
base.enable_input_require_grads()
base.config.use_cache = False
lora = cfg["lora"]
model = get_peft_model(
    base,
    LoraConfig(
        r=lora["rank"],
        lora_alpha=lora["alpha"],
        lora_dropout=lora["dropout"],
        target_modules="all-linear",
        task_type="CAUSAL_LM",
    ),
)
for parameter in model.parameters():
    if parameter.requires_grad:
        parameter.data = parameter.data.float()
model.print_trainable_parameters()


def pad_right(rows: list[list[int]], value: int) -> torch.Tensor:
    width = max(len(row) for row in rows)
    return torch.tensor([row + [value] * (width - len(row)) for row in rows], device=device)


@torch.no_grad()
def decode(
    examples, beams: int, samples: int = 0, batch_size: int | None = None
) -> tuple[list[list[str]], list[list[float]]]:
    """Beam search (samples=0) or sampling; returns programs and length-normalized log-probs."""
    model.eval()
    model.config.use_cache = True
    tokenizer.padding_side = "left"
    outputs, all_scores = [], []
    width_per = samples or beams
    batch_size = batch_size or cfg["generation"]["eval_batch_size"]
    for start in range(0, len(examples), batch_size):
        batch = examples[start : start + batch_size]
        prompts = [prompt_ids(example) for example in batch]
        width = max(len(p) for p in prompts)
        input_ids = torch.tensor(
            [[tokenizer.pad_token_id] * (width - len(p)) + p for p in prompts], device=device
        )
        attention = torch.tensor(
            [[0] * (width - len(p)) + [1] * len(p) for p in prompts], device=device
        )
        options = (
            {
                "do_sample": True,
                "num_beams": 1,
                "temperature": cfg["sampling"]["temperature"],
                "top_p": cfg["sampling"]["top_p"],
            }
            if samples
            else {"do_sample": False, "num_beams": beams}
        )
        with torch.autocast(device.type, dtype=torch.float16, enabled=use_fp16):
            generated = model.generate(
                input_ids=input_ids,
                attention_mask=attention,
                max_new_tokens=cfg["model"]["max_new_tokens"],
                num_return_sequences=width_per,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                output_scores=True,
                return_dict_in_generate=True,
                **options,
            )
        new_tokens = generated.sequences[:, width:]
        if beams > 1 and not samples:
            # Beam search already returns the length-normalized log-probability of each sequence.
            sequence_scores = generated.sequences_scores.float().tolist()
        else:
            transition = model.compute_transition_scores(
                generated.sequences, generated.scores, normalize_logits=True
            )
            valid = new_tokens[:, : transition.shape[1]] != tokenizer.pad_token_id
            lengths = valid.sum(dim=1).clamp(min=1)
            sequence_scores = (
                transition.float().masked_fill(~valid, 0).sum(dim=1) / lengths
            ).tolist()
        texts = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
        programs = [extract_program(text) for text in texts]
        for i in range(0, len(programs), width_per):
            outputs.append(programs[i : i + width_per])
            all_scores.append([round(x, 5) for x in sequence_scores[i : i + width_per]])
    model.config.use_cache = False
    return outputs, all_scores


def score(examples, programs) -> tuple[list, dict]:
    scores = [
        score_program_prediction(record, program, example.fact_texts)
        for record, example, program in zip(dev, examples, programs, strict=True)
    ]
    return scores, summarize_scores(scores)


training = cfg["training"]
step_size = training["micro_batch_size"] * training["gradient_accumulation"]
examples_per_epoch = len(train) + len(self_training_examples(train, verified, epoch=0))
steps_per_epoch = -(-examples_per_epoch // step_size)
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=training["learning_rate"]
)
scheduler = get_linear_schedule_with_warmup(
    optimizer, int(training["warmup_ratio"] * steps_per_epoch * epochs), steps_per_epoch * epochs
)
scaler = torch.amp.GradScaler(enabled=use_fp16)
if REUSE_ADAPTER:
    history = json.loads((model_dir / "history.json").read_text())["history"]
else:
    history = []
best = -1
model_dir.mkdir(parents=True, exist_ok=True)
run_started = time.time()
report_every = max(1, steps_per_epoch // 10)
for epoch in range(0 if REUSE_ADAPTER else epochs):
    model.train()
    # Every gold example (as in Notebook 17) plus one verified program per record, rotated by epoch.
    gold_examples = build_training_examples(train, facts_per_input=facts, seed=seed + epoch)
    extra_examples = self_training_examples(train, verified, epoch=epoch)
    examples = [*gold_examples, *extra_examples]
    assert len(examples) == examples_per_epoch
    print(
        {"epoch": epoch, "gold_examples": len(gold_examples), "extra_examples": len(extra_examples)}
    )
    random.Random(seed * 1_000_003 + epoch).shuffle(examples)
    features = [training_features(example) for example in examples]
    started, losses = time.time(), []
    print(f"\nEpoch {epoch + 1}/{epochs}", flush=True)
    for step, start in enumerate(range(0, len(features), step_size), start=1):
        batch = features[start : start + step_size]
        tokens = sum(sum(label != IGNORE_INDEX for label in labels) for _, labels in batch)
        optimizer.zero_grad(set_to_none=True)
        step_loss = 0.0
        for chunk_start in range(0, len(batch), training["micro_batch_size"]):
            chunk = batch[chunk_start : chunk_start + training["micro_batch_size"]]
            input_ids = pad_right([ids for ids, _ in chunk], tokenizer.pad_token_id)
            labels = pad_right([labels for _, labels in chunk], IGNORE_INDEX)
            attention = pad_right([[1] * len(ids) for ids, _ in chunk], 0)
            with torch.autocast(device.type, dtype=torch.float16, enabled=use_fp16):
                logits = model(input_ids=input_ids, attention_mask=attention).logits
            shift_logits = logits[:, :-1].float()
            shift_labels = labels[:, 1:]
            loss = (
                torch.nn.functional.cross_entropy(
                    shift_logits.reshape(-1, shift_logits.size(-1)),
                    shift_labels.reshape(-1),
                    ignore_index=IGNORE_INDEX,
                    reduction="sum",
                )
                / tokens
            )
            scaler.scale(loss).backward()
            step_loss += loss.item()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        losses.append(step_loss)
        if step % report_every == 0 or step == steps_per_epoch:
            elapsed = time.time() - started
            print(
                f"  step {step}/{steps_per_epoch} | loss {np.mean(losses[-report_every:]):.4f} | "
                f"elapsed {clock(elapsed)} | epoch left ~{clock(elapsed / step * (steps_per_epoch - step))}",
                flush=True,
            )
    train_seconds = time.time() - started
    greedy = [candidates[0] for candidates in decode(retrieved_examples, beams=1)[0]]
    _, summary = score(retrieved_examples, greedy)
    history.append(
        {
            "epoch": epoch,
            "train_loss": float(np.mean(losses)),
            "train_seconds": round(train_seconds, 1),
            "epoch_seconds_with_dev": round(time.time() - started, 1),
            **{f"dev_greedy_{k}": v for k, v in summary.items() if k != "records"},
        }
    )
    if summary["execution_correct"] > best:
        best = summary["execution_correct"]
        model.save_pretrained(model_dir / "best")
        tokenizer.save_pretrained(model_dir / "best")
    if device.type == "cuda":
        history[-1]["peak_gpu_gb"] = round(torch.cuda.max_memory_allocated() / 1024**3, 2)
    (model_dir / "history.json").write_text(
        json.dumps({"environment": environment, "history": history}, indent=2)
    )
    print(
        f"Epoch {epoch + 1} done: {history[-1]} | total {clock(time.time() - run_started)}",
        flush=True,
    )
display(pd.DataFrame(history))

## 4. Best adapter: 4-beam dev decode (top beam)

In [ ]:
# Free the training model before loading the best adapter for decoding.
del model, base, optimizer
gc.collect()
torch.cuda.empty_cache()
model = PeftModel.from_pretrained(
    AutoModelForCausalLM.from_pretrained(
        pretrained, torch_dtype=torch.float16 if use_fp16 else torch.float32
    ),
    model_dir / "best",
).to(device)
model.config.use_cache = False
started = time.time()
beam_candidates, beam_scores = decode(retrieved_examples, beams=num_beams)
print(
    f"decoded {len(beam_candidates)} records with {num_beams} beams in {clock(time.time() - started)}"
)
assert [record["qa"]["program"] for record in train] == gold_programs_before
top_beam = [options[0] for options in beam_candidates]
top_scores, summary = score(retrieved_examples, top_beam)
path = predictions_dir / f"{st['experiment_name']}{suffix}_dev_retrieved_top_beam_predictions.jsonl"
path.parent.mkdir(parents=True, exist_ok=True)
with path.open("w", encoding="utf-8") as handle:
    for example, record_score, beams_, bscores in zip(
        retrieved_examples, top_scores, beam_candidates, beam_scores, strict=True
    ):
        row = {
            **vars(record_score),
            "fact_keys": example.fact_keys,
            "beam_candidates": beams_,
            "beam_scores": bscores,
            "commit": commit,
        }
        handle.write(json.dumps(row) + "\n")
print(json.dumps(summary, indent=2))

## 5. Paired comparison with Notebook 17 (625/883)

In [ ]:
baseline_path = (
    None if SMOKE_TEST else find_artifact(st["baseline_predictions"], st["baseline_sha256"])
)
if baseline_path is None:
    print("smoke test: no baseline comparison")
else:
    baseline = [json.loads(line) for line in baseline_path.read_text().splitlines()]
    assert [row["record_id"] for row in baseline] == [s.record_id for s in top_scores]
    old = np.array([row["execution_correct"] for row in baseline])
    new = np.array([s.execution_correct for s in top_scores])
    assert old.sum() == st["baseline_correct"]
    fixed, broken = int((new & ~old).sum()), int((old & ~new).sum())
    p_value = binomtest(fixed, fixed + broken).pvalue if fixed + broken else 1.0
    comparison = {
        "baseline_execution_correct": int(old.sum()),
        "baseline_program_correct": int(sum(row["program_correct"] for row in baseline)),
        "execution_correct": int(new.sum()),
        "execution_accuracy": round(float(new.mean()), 4),
        "program_correct": int(sum(s.program_correct for s in top_scores)),
        "program_accuracy": round(float(np.mean([s.program_correct for s in top_scores])), 4),
        "fixed": fixed,
        "broken": broken,
        "net": fixed - broken,
        "mcnemar_exact_p": round(float(p_value), 6),
        "meets_rule": bool(fixed > broken and p_value < 0.05),
    }
    print(json.dumps(comparison, indent=2))
    report = {
        "environment": environment,
        "audit": audit,
        "history": history,
        "dev_top_beam": summary,
        "comparison": comparison,
        "examples_per_epoch": examples_per_epoch,
        "verified_sha256": st["verified_sha256"],
    }
    (predictions_dir / f"{st['experiment_name']}_dev_report.json").write_text(
        json.dumps(report, indent=2, default=str)
    )